<a href="https://colab.research.google.com/github/Amit-M-14/DL-01/blob/main/PneumoniaDetectionCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from logging import exception
import os
from google.colab import userdata

try:
  os.environ['KAGGLE_TOKEN'] = userdata.get('KAGGLE_TOKEN')
  print("Kaggle API token secured and loaded successfully")

  print("Downloading dataset for kaggle")
  !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

  print("unzipping images....")

  !unzip -q chest-xray-pneumonia.zip -d dataset/

  print("Data download and extraction completed successfully")

except exception as e :
  print(f"error loading kaggle token : {e}")
  print(f"please ensure you have added the KAGGLE TOKEN TO GET THE colab secrets")

Kaggle API token secured and loaded successfully
Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [02:17<00:00, 18.0MB/s]

unzipping images....
Data download and extraction completed successfully


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import dataloader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"successfully connected to device {device}")

data_transforms = transforms.Compose([
    transforms.Resize((244,244)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229, 0.224, 0.225])
])

train_dir = 'dataset/chest_xray/train'
test_dir = 'dataset/chest_xray/test'

train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms)

train_loader = dataloader.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = dataloader.DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Data pipeline read! Loaded {len(train_dataset)} training images.")

successfully connected to device cuda
Data pipeline read! Loaded 5216 training images.


In [ ]:
import torch.nn as nn
from torchvision import models

print("Initializing the prrtrained RestNet50 Architecture...")
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

for param in model.parameters():
  param.requires_grad = False

  num_ftrs = model.fc.in_features

  model.fc = nn.Linear(num_ftrs, 2)

  model = model.to(device)

print("Model customizatiion complete. Ready for final layer training")

Initializing the prrtrained RestNet50 Architecture...
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s]


Model customizatiion complete. Ready for final layer training


In [ ]:
import torch.optim as optim
import time

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

epochs = 5
print(f"beginning optimization routiine on {device} for {epochs} epochs")

for epoch in range(epochs):
  start_time = time.time()
  model.train()

  running_loss = 0.0
  correct_predictions = 0
  total_predictions = 0

  for inputs, label in train_loader:
      inputs, label = inputs.to(device), label.to(device)

      optimizer.zero_grad()

      outputs = model(inputs)
      loss = criterion(outputs, label)
      loss.backward()
      optimizer.step()

      running_loss += loss.item() * inputs.size(0)
      _, predicted = torch.max(outputs, 1)
      total_predictions += label.size(0)
      correct_predictions += (predicted == label).sum().item()

      epoch_loss = running_loss / len(train_loader.dataset)
      epoch_acc = (correct_predictions / total_predictions) * 100
      epoch_time = time.time() - start_time

      print(f"Epoch [{epoch+1}/{epochs}] | Process Time: {epoch_time:.1f}s | Train Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("\nTraining Phase Finalized! The diagnostic model is ready for testing.")

beginning optimization routiine on cuda for 5 epochs
Epoch [1/5] | Process Time: 1.5s | Train Loss: 0.0042 | Accuracy: 59.38%
Epoch [1/5] | Process Time: 2.0s | Train Loss: 0.0079 | Accuracy: 68.75%
Epoch [1/5] | Process Time: 2.6s | Train Loss: 0.0115 | Accuracy: 70.83%
Epoch [1/5] | Process Time: 3.2s | Train Loss: 0.0147 | Accuracy: 71.88%
Epoch [1/5] | Process Time: 3.7s | Train Loss: 0.0175 | Accuracy: 73.75%
Epoch [1/5] | Process Time: 4.3s | Train Loss: 0.0201 | Accuracy: 75.52%
Epoch [1/5] | Process Time: 4.8s | Train Loss: 0.0232 | Accuracy: 75.89%
Epoch [1/5] | Process Time: 5.4s | Train Loss: 0.0273 | Accuracy: 75.00%
Epoch [1/5] | Process Time: 5.9s | Train Loss: 0.0301 | Accuracy: 75.35%
Epoch [1/5] | Process Time: 6.4s | Train Loss: 0.0327 | Accuracy: 75.94%
Epoch [1/5] | Process Time: 7.3s | Train Loss: 0.0368 | Accuracy: 74.43%
Epoch [1/5] | Process Time: 8.1s | Train Loss: 0.0396 | Accuracy: 74.48%
Epoch [1/5] | Process Time: 8.9s | Train Loss: 0.0426 | Accuracy: 74.04

In [ ]:
!pip install gradio -q

import gradio as gr
from PIL import Image
import torch.nn.functional as F
import torch

model.eval()

def predict_xray(img):
    img = img.convert('RGB')

    img_tensor = data_transforms(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img_tensor)
        probabilities = F.softmax(output[0], dim=0)

    class_names = ['Normal', 'Pneumonia']

    return {class_names[i]: float(probabilities[i]) for i in range(2)}

interface = gr.Interface(
    fn=predict_xray, # The function we just wrote
    inputs=gr.Image(type="pil"), # A drag-and-drop image upload box
    outputs=gr.Label(num_top_classes=2), # A clean label showing the prediction
    title="🩺 Automated Chest X-Ray Diagnostics",
    description="Upload a patient's chest X-ray to instantly detect the presence of Pneumonia using a fine-tuned Deep Learning model.",
    allow_flagging="never"
)

print("Launching the User Interface...")
interface.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Launching the User Interface...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5411a1ea7a1c74f09d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5411a1ea7a1c74f09d.gradio.live
